In [1]:
from opt_targeted_transfers import BinaryGapTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [2]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [3]:
tt = BinaryGapTargetedTransfers(c_bar=2.15, n_regressors=5)

In [4]:
# Nuisance parameter estimation
# Fit conditional improvement regressors for different transfer values
tt.fit(train_dataset, validation_dataset)

Fitting conditional gap improvement for transfer size 0.01


100%|██████████| 300/300 [00:03<00:00, 81.59it/s, val loss=0.89] 


Fitting conditional gap improvement for transfer size 0.545


100%|██████████| 300/300 [00:03<00:00, 93.24it/s, val loss=0.918] 


Fitting conditional gap improvement for transfer size 1.08


100%|██████████| 300/300 [00:03<00:00, 91.92it/s, val loss=0.955] 


Fitting conditional gap improvement for transfer size 1.615


100%|██████████| 300/300 [00:03<00:00, 85.38it/s, val loss=0.979]


Fitting conditional gap improvement for transfer size 2.15


100%|██████████| 300/300 [00:03<00:00, 81.77it/s, val loss=0.978]


In [5]:
# Precomputation for policy optimization step.
tt.optimize_transfers_for_budget_grid(test_covariate_dataset, budgets=[0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.15])

In [6]:
# Set budget and run policy optimization step for that budget.
# Policy optimization step returns transfer amount for each unit in the test set.
# Note that budget must lie in the set of budgets used in the precomputation step.
tt.set_budget(budget=1.0)
assignments = tt.run_opt(test_covariate_dataset)

In [7]:
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.04450745063123372,
 'post_transfer_poverty_rate': 0.16222612210863313,
 'policy_cost_per_capita': 0.9998800675751829,
 'budget': 1.0,
 'policy_type': 'binary_gap',
 'd': 2}

In [8]:
# Can try a different budget without redoing the fit step and precomputation step.
# Note that budget must lie in the set of budgets used in the precomputation step.
# Setting the budget will clear assignments attribute.
tt.set_budget(1.5)
tt.run_opt(test_covariate_dataset)
res = tt.evaluate(test_dataset)
res


{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.008028498371086004,
 'post_transfer_poverty_rate': 0.02303141832593607,
 'policy_cost_per_capita': 1.4999873804655806,
 'budget': 1.5,
 'policy_type': 'binary_gap',
 'd': 2}